# Manual PyTorch Training Pipeline
This notebook implements a complete binary classification training pipeline in PyTorch **from scratch** (without using high-level modules like `nn.Module` or `torch.optim`).

We will perform:
1. Data preprocessing (loading, cleaning, splitting, scaling, and label encoding).
2. Custom tensor initialization for weights and biases.
3. Manual definition of the forward pass, Sigmoid activation, and Binary Cross-Entropy (BCE) loss.
4. A manual training loop utilizing PyTorch's automatic differentiation (`loss.backward()`) and gradient descent updates.

## 1. Importing Libraries

In [1]:
# Importing libraries for preprocessing and numerical computations
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## 2. Loading the Dataset
We load the Breast Cancer Detection dataset directly from a raw CSV file.

In [2]:
# Loading the dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## 3. Exploratory Data Analysis & Data Cleaning

In [3]:
# Display initial dataset dimensions
print(f"No of rows : {df.shape[0]} ")
print(f"No of columns : {df.shape[1]} ")

No of rows : 569 
No of columns : 33 


In [4]:
# Drop columns that are not predictive features (id and empty Unnamed: 32)
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [5]:
# Display cleaned dataframe head
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
# Verify new dimensions
print(f"No of rows : {df.shape[0]} ")
print(f"No of columns : {df.shape[1]} ")

No of rows : 569 
No of columns : 31 


## 4. Preprocessing: Splitting and Feature Scaling
We split the dataset into training (80%) and testing (20%) sets, and scale features to have zero mean and unit variance.

In [7]:
# Train-test split (Note: label is in column index 0; features in column index 1 onwards)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

In [8]:
# Scale the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 5. Preprocessing: Label Encoding
We encode the categorical class labels (Malignant/Benign) to numerical labels (1/0).

In [9]:
# View raw training labels before encoding
y_train

68     B
181    M
63     B
248    B
60     B
      ..
71     B
106    B
270    B
435    M
102    B
Name: diagnosis, Length: 455, dtype: object

In [10]:
# Fit and transform labels to binary integer targets
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
# View encoded training labels
y_train

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,

## 6. Converting Data to PyTorch Tensors
We convert NumPy arrays to PyTorch Tensors so that gradients can be tracked.

In [12]:
# Convert to PyTorch tensors (cast to float32/float64)
X_train_tensor = torch.from_numpy(X_train).double()
X_test_tensor = torch.from_numpy(X_test).double()
y_train_tensor = torch.from_numpy(y_train).double()
y_test_tensor = torch.from_numpy(y_test).double()

In [13]:
# Inspect tensor shape of training features
X_train_tensor.shape

torch.Size([455, 30])

In [14]:
# Inspect tensor shape of training labels
y_train_tensor.shape

torch.Size([455])

## 7. Defining the Custom Single-Layer Neural Network
We define `MySimpleNN` containing manual weights, bias, forward propagation, and Binary Cross-Entropy loss computation.

In [15]:
# Defining the model class from scratch
class MySimpleNN():
    def __init__(self, X):
        # Initialize weights randomly with require_grad=True to enable backpropagation
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        # Initialize bias to zeros with require_grad=True
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        # Linear transformation z = X * w + b
        z = torch.matmul(X, self.weights) + self.bias
        # Squash to [0, 1] range using Sigmoid function
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y):
        # Clamp predictions to avoid log(0) which results in NaN values
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        
        # Reshape y to (N, 1) to avoid PyTorch dimensions broadcasting bug (critical bug fix)
        y_reshaped = y.view(-1, 1)
        
        # Calculate Binary Cross-Entropy loss manually
        loss = -(y_reshaped * torch.log(y_pred) + (1 - y_reshaped) * torch.log(1 - y_pred)).mean()
        return loss

## 8. Training Configuration

In [16]:
# Set learning rate and training epochs
learning_rate = 0.1
epochs = 25

## 9. Executing the Training Loop
We run the loop: compute forward pass -> calculate loss -> call `loss.backward()` to compute gradients -> perform manual gradient descent -> clear gradients manually for the next iteration.

In [17]:
# Create the model instance
model = MySimpleNN(X_train_tensor)

# Start manual training loop
for epoch in range(epochs):
    # 1. Forward Pass
    y_pred = model.forward(X_train_tensor)

    # 2. Loss Calculation
    loss = model.loss_function(y_pred, y_train_tensor)

    # 3. Backward Pass (Computes gradients dLoss/dw and dLoss/db)
    loss.backward()

    # 4. Parameters Update (Using no_grad to prevent updating parameters tracking history)
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # 5. Reset Gradients manually (Important: PyTorch accumulates gradients by default)
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # Print loss value every epoch
    print(f'Epoch: {epoch + 1:02d} | Loss: {loss.item():.5f}')

Epoch: 01 | Loss: 0.49446
Epoch: 02 | Loss: 0.47997
Epoch: 03 | Loss: 0.46582
Epoch: 04 | Loss: 0.45199
Epoch: 05 | Loss: 0.43851
Epoch: 06 | Loss: 0.42536
Epoch: 07 | Loss: 0.41256
Epoch: 08 | Loss: 0.40010
Epoch: 09 | Loss: 0.38801
Epoch: 10 | Loss: 0.37627
Epoch: 11 | Loss: 0.36491
Epoch: 12 | Loss: 0.35392
Epoch: 13 | Loss: 0.34332
Epoch: 14 | Loss: 0.33311
Epoch: 15 | Loss: 0.32264
Epoch: 16 | Loss: 0.31092
Epoch: 17 | Loss: 0.29969
Epoch: 18 | Loss: 0.28894
Epoch: 19 | Loss: 0.27866
Epoch: 20 | Loss: 0.26885
Epoch: 21 | Loss: 0.25949
Epoch: 22 | Loss: 0.25057
Epoch: 23 | Loss: 0.24207
Epoch: 24 | Loss: 0.23398
Epoch: 25 | Loss: 0.22628


## 10. Inspecting Final Learned Bias

In [18]:
# Print the final bias parameter value
print("Final Bias:", model.bias.item())

Final Bias: -0.08441577812978021


## 11. Model Evaluation & Accuracy Measurement
We compute predictions on unseen test features and compare them to actual test targets to measure classification accuracy.

In [19]:
# Model Evaluation
with torch.no_grad():
    # Generate prediction probabilities on the test set
    test_preds = model.forward(X_test_tensor)
    
    # Set decision threshold at 0.5 (classes squashed to binary classification predictions)
    binary_preds = (test_preds > 0.5).float()
    
    # Fix broadcasting issue by squeezing binary_preds shape from (N, 1) to (N,) to match y_test_tensor (critical bug fix)
    accuracy = (binary_preds.squeeze() == y_test_tensor).float().mean()
    
    print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')

Test Accuracy: 95.61%
